# Lesson 0: Reshape Texas sales from wide to long

Input: `sales_train_evaluation.csv`, daily unit sales with one column per day.

Output: `tx_sales_long.parquet`, one row per product, store, and day for the 3 Texas stores.

Every later stage of the project loads this file instead of the raw CSV.

## Check the environment

Confirms the notebook runs on the project's virtual environment with the expected package versions (pandas 3.0.6, duckdb 1.5.5, pyarrow 25.0.1).

A wrong environment would silently use different package versions and could change results.

In [1]:
import sys, pandas, duckdb, pyarrow
print(sys.executable)
print(pandas.__version__, duckdb.__version__, pyarrow.__version__)

/Users/rajkandula/projects/walmart-demand-forecasting/.venv/bin/python
3.0.6 1.5.5 25.0.1


## Load the raw sales file

The file has 30,490 rows (3,049 products × 10 stores) and 1,947 columns: 6 ID columns plus one column per day, `d_1` to `d_1941`.

`mb()` measures a table's memory in MB. It's used at each step below to track the effect of dtype changes.

In [2]:
import sys
sys.path.append("..")
import pandas as pd
from src.config import RAW_DIR, PROCESSED_DIR

sales = pd.read_csv(RAW_DIR / "sales_train_evaluation.csv")
print(sales.shape)

def mb(df):
    return df.memory_usage(deep=True).sum() / 1e6

(30490, 1947)


## Keep only Texas

The project covers the 3 Texas stores, so I filter before reshaping. Reshaping all 10 stores first would build a much larger table only to drop 70% of it.

Expected: 9,147 rows (3,049 products × 3 stores). Memory before any dtype changes: 143.06 MB, almost all of it the 1,941 `int64` day columns.

In [3]:
tx = sales[sales["state_id"] == "TX"]
assert tx.shape == (9147, 1947)
print("Wide TX, default dtypes:", mb(tx), "MB")

Wide TX, default dtypes: 143.056716 MB


## Convert ID columns to category

Category stores each distinct value once, plus a small code per row. It saved only 0.55 MB here (143.06 to 142.51 MB), because the ID columns are about 1 MB of the wide table, and `id` and `item_id` have almost no repeated values.

The payoff comes after the reshape, when each ID value repeats 1,941 times.

In [4]:
id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
tx[id_cols] = tx[id_cols].astype("category")
print("Wide TX, category ids:", mb(tx), "MB")
print(tx[id_cols].dtypes)

Wide TX, category ids: 142.50901 MB
id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
dtype: object


## Safety checks before reshaping

Total units sold across all Texas rows and days: 19,228,405. This is checked again after the reshape to prove no sales were lost or duplicated.

Largest single daily value: 634. Sales will be stored as `int16`, which holds up to 32,767, so the conversion is safe. The assert stops the notebook if that ever stops being true.

634 units of one product in one store on one day is an outlier to investigate in Stage 2.

In [5]:
day_cols = [c for c in tx.columns if c.startswith("d_")]
print(len(day_cols))
wide_total = tx[day_cols].sum().sum()
print(wide_total)
max_value = tx[day_cols].max().max()
assert max_value <= 32767
print(max_value)

1941
19228405
634


## Reshape wide to long

`pd.melt` turns each of the 1,941 day columns into rows, repeating the 6 ID columns on every row.

Result: 17,754,327 rows (9,147 × 1,941) and 8 columns (6 IDs, `d`, `sales`). Memory jumps to 525.12 MB.

The per-column breakdown shows where it goes: `d` is 240.65 MB as repeated text and `sales` is 142.03 MB as `int64`. The ID columns stayed category through the reshape, which keeps them at about 142 MB combined. `d` and `sales` are what the next step shrinks.

In [6]:
tx_long = pd.melt(tx, id_vars=id_cols, var_name="d", value_name="sales")
print(tx_long.shape)
print(tx_long.dtypes)
print("Long, before dtype changes:", mb(tx_long), "MB")
print(tx_long.memory_usage(deep=True) / 1e6)
assert tx_long.shape == (17754327, 8)

(17754327, 8)
id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
d                str
sales          int64
dtype: object
Long, before dtype changes: 525.12459 MB
Index         0.000132
id           35.845897
item_id      35.572285
dept_id      17.754445
cat_id       17.754373
store_id     17.754364
state_id     17.754338
d           240.654140
sales       142.034616
dtype: float64


## Shrink data types and verify

`sales` becomes `int16` (2 bytes per value instead of 8), and `d` becomes category (1,941 distinct labels, each stored once, instead of text on every row).

Memory drops from 525.12 MB to 213.48 MB, a 59% reduction.

The total-units assert passing means the sales column still sums to 19,228,405, so the reshape and the type change kept every unit.

In [7]:
tx_long["sales"] = tx_long["sales"].astype("int16")
tx_long["d"] = tx_long["d"].astype("category")
assert tx_long["sales"].sum() == wide_total
print(tx_long.dtypes)
print("Long, after dtype changes:", mb(tx_long), "MB")

id          category
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
d           category
sales          int16
dtype: object
Long, after dtype changes: 213.479452 MB


## Save to Parquet

Parquet keeps the `category` and `int16` types, so later notebooks load the cleaned table directly instead of repeating these steps.

File sizes: Parquet 68.47 MB (Texas, long), raw CSV 121.74 MB (all 10 stores, wide). These aren't directly comparable, because the files differ in format, shape, and number of stores. The long format repeats 6 ID values on every row, which makes it bigger to store than a wide file of the same data.

In [8]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / "tx_sales_long.parquet"
tx_long.to_parquet(out_path, index=False)
print("Parquet:", out_path.stat().st_size / 1e6, "MB")
print("CSV:", (RAW_DIR / "sales_train_evaluation.csv").stat().st_size / 1e6, "MB")

Parquet: 68.47012 MB
CSV: 121.736518 MB


## Results

| Stage | Memory (MB) |
|---|---|
| Wide TX, default dtypes | 143.06 |
| Wide TX, category IDs | 142.51 |
| Long, before dtype changes | 525.12 |
| Long, after dtype changes | 213.48 |

- Rows: 9,147 wide, 17,754,327 long
- Total units sold, checked before and after: 19,228,405
- Largest single daily value: 634 (fits `int16`)
- Parquet file (Texas, long): 68.47 MB. Raw CSV (all 10 stores, wide): 121.74 MB

Takeaway: the reshape more than tripled memory, and two dtype changes brought it back down by 59%, with all 19,228,405 units verified. The Parquet file is the starting point for Lesson 1.